# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from google import genai

In [6]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AIza') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
openai = genai

API key looks good so far


In [7]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [15]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# 1. Reload the environment variables just to be safe
load_dotenv(override=True)

# 2. Build the OpenAI-compatible client pointing to Google
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY")
)

# Make sure your model is set to Gemini!
MODEL = "gemini-2.5-flash"

def select_relevant_links(url):
    # 3. Use 'client.chat' instead of 'openai.chat'
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

# Test it out!
# print(select_relevant_links("https://edwarddonner.com"))

In [16]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'educational program',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'educational program',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'related project/company',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'social media - professional',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media - professional',
   'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media - professional',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [19]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [20]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash
Found 5 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'offerings page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'product/service page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [21]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 17 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'about page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'company news page', 'url': 'https://huggingface.co/blog'},
  {'type': 'business solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'product page', 'url': 'https://huggingface.co/models'},
  {'type': 'product page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learning page', 'url': 'https://huggingface.co/learn'},
  {'type': 'company news page', 'url': 'https://huggingface.c

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/gemma-4-31B-it
Updated
2 days ago
•
287k
•
827
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
12 days ago
•
524k
•
2.29k
CohereLabs/cohere-transcribe-03-2026
Updated
2 days ago
•
96.6k
•
783
baidu/Qianfan-OCR
Updated
9 days ago
•
36.6k
•
917
prism-ml/Bonsai-8B-gguf
Updated
5 days ago
•
32.9k
•
380
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.83k
Wan2.2 14B Preview
🐌
1.83k
generate a video fr

In [25]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':


brochure_system_prompt = """
 You are an assistant that analyzes the contents of several relevant pages from a company website
 and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
 Respond in markdown without code blocks.
 Include details of company culture, customers and careers/jobs if you have the information.
 """


In [26]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [27]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 8 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n2 days ago\n•\n287k\n•\n828\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n12 days ago\n•\n524k\n•\n2.29k\nCohereLabs/cohere-transcribe-03-2026\nUpdated\n2 days ago\n•\n96.6k\n•\n783\nbaidu/Qianfan-OCR\nUpdated\n9 days ago\

In [30]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [31]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 10 relevant links


Hugging Face: The AI Party Where Everyone's Invited (Especially the Robots)

Ever feel like the future of AI is happening somewhere else? Well, it's probably here, at Hugging Face! We're not just a platform; we're the bustling, brainy, and slightly overcaffeinated heart of the machine learning universe. Think of us as the ultimate social network, but for AI models, datasets, and applications – because even algorithms need friends (and maybe a few thousand collaborators).

**What We Do (With More Zest Than a CPU Overclocking):**
We host more digital brains than a sci-fi convention (over 2 million models, to be exact!). Want to teach a computer to write poetry? Build an app that turns your selfies into Renaissance masterpieces? Or perhaps just find the perfect dataset to prove cats are secretly planning world domination? We've got you covered.

*   **Models:** From the heavyweight Google Gemma 4 to the latest llama.cpp wizardry, our models are so hot, they practically come with a "do not touch" warning (but please do!). They're trending, they're smart, and they're ready for their close-up.
*   **Datasets:** With 500k+ datasets, we're like the Library of Alexandria, but for data. (And less prone to accidental burning, we hope.) Need something specific? Chances are, someone in our community already shared it.
*   **Spaces (AI Apps):** Launch your brilliant AI ideas into the stratosphere with our 1M+ applications. See an image generate a video, or edit 3D camera angles – all running on advanced compute like our magical ZeroGPU. It’s like a sandbox for geniuses, complete with extra private storage Buckets for your secret AI projects.

**Who We Serve (And Why They Hug Us Back):**

*   **The Lone Wolf AI Enthusiast:** Got a groundbreaking idea? Come play in our open-source playground. We empower individual engineers and scientists to learn, collaborate, and share their work to build an open and ethical AI future together.
*   **The Super Squad & Enterprise Titans:** Our Team & Enterprise plans offer enough security, audit logs, SSO, and token management to make a secret agent blush. We give your organization 5x more ZeroGPU quota – because we know your AI ambitions are *massive*. And with private storage, resource groups, and dedicated support, your AI secrets stay, well, *secret* (unless you want to share them, which we enthusiastically encourage!).

**Our Vibe (It's All About That Hug!):**
We believe AI shouldn't be locked in a dark room, muttering to itself. It should be out in the open, collaborating, learning, and occasionally getting a virtual high-five (or a hug!). Our community is fast-growing, passionate, and dedicated to pushing the boundaries of what's possible, all while having a darn good time. We're at the heart of the AI revolution, fostering collaboration and open-source innovation.

**Join the Fun (Before AI Takes Over All the Good Spots)!**
Whether you're looking to build, invest, or simply make your mark on the AI revolution, Hugging Face is the place to be. We’re always looking for brilliant minds to join our diverse team (check out our "Current Openings" – we promise they’re more exciting than they sound!). Come contribute to the heart of the AI revolution, where innovation gets a hug every single day.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [34]:
def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 16 relevant links


### Hugging Face: Where AI Gets a Big, Friendly Hug (and So Can You!)

Tired of AI feeling cold, distant, and locked behind opaque corporate walls? At Hugging Face, we're warming things up! We believe AI should be collaborative, accessible, and maybe even a little cuddly. We're not just a platform; we're the world's most bustling bazaar, playground, and friendly neighborhood for all things machine learning!

**What We Do (and Why You'll Love Us!)**

*   **Models Galore!** Got an AI model? We've got over **2 million** friends for it to play with! From text generation that writes your break-up notes (we don't judge!) to image-to-video tools that turn your cat's antics into a Hollywood trailer, our models do it all. We're constantly welcoming new stars like the dazzling Google Gemma 4 and even recent cool kids like GGML and llama.cpp.

*   **Datasets for Days!** Need data? We've got over **500,000** datasets, meticulously collected, curated, and occasionally debated over with fervent passion. It's like a data buffet, but without the mystery meat – just pure, delicious information for your AI to feast on.

*   **Spaces: Your AI Stage!** Want to show off your brilliant AI app? Our **1 million+** Spaces are where your creations truly shine! Think of it as YouTube for AI demos, but way smarter and less likely to show you unsolicited cat videos (unless, of course, your app *makes* cat videos, in which case, share away!).

*   **Buckets: Your AI's Home Sweet Home!** Running out of room for your AI masterpieces? Our new, AI-native Storage Buckets are here to embrace all your digital treasures. They're like a bottomless pit of secure storage, but optimized for machine learning – so your data feels right at home, nestled comfortably.

*   **The Power of Open Source!** We're all about sharing is caring! Our open-source stack means you can move faster, build bolder, and probably take over the world with AI quicker than you thought possible. Collaboration isn't just a feature; it's our love language.

**Who's Getting Hugged?**

*   **For Our Esteemed Customers (and Future Customers!):** Whether you're an AI wizard, a data whisperer, a curious coder, or an enterprise looking to sprinkle a little AI magic on your bottom line, Hugging Face is your happy place. We're here for anyone who believes "machine learning" isn't just a buzzword, but a gateway to a smarter, more efficient (and fun!) future.

*   **For Our Savvy Investors:** Looking to invest in the future? We're building it, one collaborative model, dataset, and application at a time. Join us, and let's make some AI-powered magic (and maybe a significant return) together!

*   **For Our Brilliant Recruits:** Dreaming of a career where you're literally shaping the future? Love collaborating, innovating, and occasionally debating the merits of different transformer architectures over coffee? Come join our global AI family! We offer the chance to work on cutting-edge tech, solve mind-bending problems, and get virtual hugs from millions of fellow AI enthusiasts. Your career here won't just "grow"; it'll *transFORM*! We're a community, a culture of builders, and a place where every idea gets a fair listen.

**Ready to get a Hug?**

Stop just *thinking* about AI and start *doing* AI! Dive into Hugging Face today. Explore models, create a space, or simply browse and marvel. The future of AI, built by the community, is waiting to give you a big, friendly hug!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 6 relevant links


**Hugging Face: Get Ready to Embrace the Future (No Actual Hugs Required... Usually)**

Tired of AI feeling like a lonely, dark room filled with cryptic error messages? Welcome to Hugging Face, where the future of machine learning is built in public, with friends, and maybe a few million models.

**Who We Are:**
We're not just a platform; we're *the* AI community. Think of us as the world's largest, most enthusiastic potluck for AI – everyone brings their best models, datasets, and applications, and we all feast on innovation. We're where the magic happens, and sometimes, even Google Gemma 4 drops by for a visit!

**For the AI Enthusiast (That's You!):**

*   **Models Galore:** Dive into over 2 million models, from the latest multimodal marvels to the classics. We even convinced GGML and llama.cpp to join the party – talk about a popular crowd!
*   **Data Delights:** Feast your algorithms on 500k+ datasets. Enough training data to make your GPU purr with delight (or perhaps scream, depending on your batch size).
*   **Spaces & Apps:** Bring your wildest AI dreams to life with over 1 million applications. Generate videos from images, edit 3D camera angles, or just chat with a multimodal AI (it's less awkward than talking to yourself, we promise).
*   **Open Source Power:** Accelerate your projects with our legendary open-source stack. Move faster than a neural net on caffeine!

**For the Enterprise Conqueror:**
Scale your AI ambitions with enterprise-grade security, single sign-on (because who remembers passwords anymore?), audit logs, and dedicated support. Control your data's destiny with region selection – because sometimes, your models just want to stay close to home. We make building AI so easy, your competitors will wonder if you hired a team of cyborg geniuses (you probably did, with our help!).

**Our Culture (The Human Element):**
We're a vibrant, collaborative bunch obsessed with open science and making AI accessible. We believe in sharing, learning, and creating the future, one model pull request at a time. It's a place where creativity meets cutting-edge tech, and every day is an opportunity to push the boundaries of what AI can do. If you thrive on collaboration, innovation, and a healthy dose of digital camaraderie, you'll feel right at home.

**Join the Revolution! (Careers & Investment):**
If you're an ambitious individual ready to shape the AI landscape, you've found your tribe. We're building something massive, and we need brilliant minds like yours to help us craft the next generation of intelligent systems. Investors: We're not just trending; we're *the* trend. Come invest in the community that's literally building the future.

**Ready to Get Hugged by Innovation?**
Stop staring blankly at your screen. Head over to HuggingFace.co and become part of the AI future today! Your models will thank you.

: 

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>